In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

In [2]:
matplotlib.use('tkagg')
%matplotlib auto
%matplotlib auto

Using matplotlib backend: tkagg
Using matplotlib backend: tkagg


In [3]:
data_root = '/home/fourier/GRX_humanoid/'
sdk_path = data_root + 'real data/251205_233_下蹲高度不足'
real_data_name = 'wbc_full38_0.1.csv'
real_data_path = sdk_path + '/' + real_data_name

start_id = 0
end_id = 10000
real_csv = pd.read_csv(real_data_path, skiprows=range(1, start_id + 1), nrows=end_id-start_id)

time_strings = real_csv.iloc[:, 0]
print(time_strings[:5])
real_time = [datetime.strptime(t[11:], '%H:%M:%S.%f').timestamp() for t in time_strings]
real_time_seconds = [(t - real_time[0]) for t in real_time]
real_time_seconds = np.round(real_time_seconds, decimals=4)
print(len(real_time_seconds))
print(real_time_seconds[:10])

real_data = real_csv.iloc[:, :].values
print(real_data.shape)

0    2025-12-05 16:31:19.392
1    2025-12-05 16:31:19.412
2    2025-12-05 16:31:19.432
3    2025-12-05 16:31:19.453
4    2025-12-05 16:31:19.473
Name: 2025-12-05 16:31:19.372, dtype: object
10000
[0.    0.02  0.04  0.061 0.081 0.101 0.121 0.141 0.161 0.181]
(10000, 189)


In [4]:
# parse data
ac_dim = 31
upper_num = 16
real_ang_vel = real_data[:, 1:4]
real_gravity = real_data[:, 4:7]
real_jpos = real_data[:, 7:7+ac_dim]
real_jvel = real_data[:, 7+ac_dim:7+2*ac_dim]
real_jTor = real_data[:, 7+2*ac_dim:7+3*ac_dim]
real_jTarg = real_data[:, 7+3*ac_dim:7+4*ac_dim]
real_action = real_data[:, 7+4*ac_dim:7+5*ac_dim]
real_walk_flag = real_data[:, 7+5*ac_dim:7+5*ac_dim+1]
real_velCmd = real_data[:, 7+5*ac_dim+1:7+5*ac_dim+4]
real_haCmd = real_data[:, 7+5*ac_dim+4:7+5*ac_dim+4+4]
real_upper_cmd = real_data[:, 7+5*ac_dim+4+4:7+5*ac_dim+4+4+upper_num]
real_CoM_pos = real_data[:, 7+5*ac_dim+4+4+upper_num:7+5*ac_dim+4+4+upper_num+3]

In [ ]:
# 提取指令数据
command = real_data[:, 7+5*ac_dim: 7+5*ac_dim+1+3+4+upper_num]
np.savetxt(sdk_path + '/parsed_command.csv', command, delimiter=',')

In [5]:
ppv210_joint_names = ["left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint", \
                    "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint", \
                    "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",  \
                    "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",\
                    "waist_yaw_joint", "waist_roll_joint","waist_pitch_joint", \
                    "head_yaw_joint", "head_pitch_joint",\
                    "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint", "left_elbow_pitch_joint", \
                    "left_wrist_yaw_joint", "left_wrist_pitch_joint", "left_wrist_roll_joint", \
                    "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint", "right_elbow_pitch_joint", \
                    "right_wrist_yaw_joint", "right_wrist_pitch_joint", "right_wrist_roll_joint"]
upper_joint_names = [   "head_yaw_joint", "head_pitch_joint",\
                        "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint", "left_elbow_pitch_joint", \
                        "left_wrist_yaw_joint", "left_wrist_pitch_joint", "left_wrist_roll_joint", \
                        "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint", "right_elbow_pitch_joint", \
                        "right_wrist_yaw_joint", "right_wrist_pitch_joint", "right_wrist_roll_joint"]
lower_joint_names = [   "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint", \
                        "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint", \
                        "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",  \
                        "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",\
                        "waist_yaw_joint", "waist_roll_joint","waist_pitch_joint"]

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(real_time_seconds, real_jpos[:, i+15], label=f'real_jPos')
    ax.plot(real_time_seconds, real_upper_cmd[:, i], label=f'real_upper_cmd')
    ax.plot(real_time_seconds, real_jTarg[:, i+15], label=f'real_jTarg')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
    # ax2 = ax.twinx()
    # ax2.plot(sim_velCmd[:, 0], label=f'Vx', linestyle='--')
    # ax2.plot(sim_velCmd[:, 1], label=f'Vy', linestyle='--')
    # ax2.plot(sim_velCmd[:, 2], label=f'Wz', linestyle='--')
    # ax2.plot(real_time, real_haCmd[:, 0], label=f'Hgt_real', color='lightblue', linestyle='--')
    # ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Upper Body Joint Positions and Targets', fontsize=16)
plt.show()

In [7]:
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(real_time_seconds, real_jpos[:, i], label=f'real_jPos')
    ax.plot(real_time_seconds, real_jTarg[:, i], label=f'real_jTarg')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(real_time_seconds, real_haCmd[:, 0], label=f'height cmd', linestyle='--')
    ax2.plot(real_time_seconds, real_haCmd[:, 1], label=f'yaw cmd', linestyle='--')
    ax2.plot(real_time_seconds, real_haCmd[:, 2], label=f'roll cmd', linestyle='--')
    ax2.plot(real_time_seconds, real_haCmd[:, 3], label=f'pitch cmd', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Upper Body Joint Positions and Targets', fontsize=16)
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(real_time_seconds, real_jvel[:, i+15], label=f'real_jVel')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(real_time_seconds, real_upper_cmd[:, i], label=f'real_upper_cmd', linestyle='--', color='orange')
plt.tight_layout()
plt.suptitle('Upper Body Joint Positions and Targets', fontsize=16)
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(real_time_seconds, real_jTor[:, i+15], label=f'real_jTor')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(real_time_seconds, real_upper_cmd[:, i], label=f'real_upper_cmd', linestyle='--', color='orange')
plt.tight_layout()
plt.suptitle('Upper Body Joint Positions and Targets', fontsize=16)
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(real_time_seconds, real_jpos[:, i], label=f'real_jPos')
    ax.plot(real_time_seconds, real_jTarg[:, i], label=f'real_jTarg', linestyle='--')
    # ax.plot(real_time, real_jvel[:, i], label=f'real_jVel', color='orange', linestyle=':')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
    # ax2 = ax.twinx()
    # ax2.plot(real_time, real_jTor[:, i], label=f'real_jTor', color='green', linestyle='-.')
    # ax2.plot(real_time, real_ang_vel[:, 0], label=f'Wx', color='lightblue', linestyle='--')
    # ax2.plot(real_time, real_ang_vel[:, 1], label=f'Wy', color='lightgreen', linestyle='--')
    # ax2.plot(real_time, real_ang_vel[:, 2], label=f'Wz', color='lightpink', linestyle='--')
    # ax2.plot(real_time, real_gravity[:, 0], label=f'grav0', color='lightblue', linestyle='--')
    # ax2.plot(real_time, real_gravity[:, 1], label=f'grav1', color='lightgreen', linestyle='--')
    # ax2.plot(real_time, real_gravity[:, 2], label=f'grav2', color='lightpink', linestyle='--')
    # ax2.plot(sim_velCmd[:, 0], label=f'Vx', linestyle='--')
    # ax2.plot(sim_velCmd[:, 1], label=f'Vy', linestyle='--')
    # # ax2.plot(sim_velCmd[:, 2], label=f'Wz', linestyle='--')
    # ax2.plot(real_time, real_haCmd[:, 0], label=f'Hgt_real', color='lightblue', linestyle='--')
    # ax2.plot(real_time, real_haCmd[:, 1], label=f'yaw_targ', color='blue', linestyle='--')
    # ax2.plot(real_time, real_haCmd[:, 2], label=f'roll_targ', color='lightgreen', linestyle='--')
    # ax2.plot(real_time, real_haCmd[:, 3], label=f'pitch_targ', color='lightpink', linestyle='--')
    # ax2.plot(real_time, real_CoM_pos[:, 0], label=f'CoM_x', color='lightcoral', linestyle='--')
    # ax2.plot(real_time, real_CoM_pos[:, 1], label=f'CoM_y', color='lightgreen', linestyle='--')
    # ax2.plot(real_time, real_CoM_pos[:, 2], label=f'CoM_z', color='lightpink', linestyle='--')
    # ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Lower Body Joint Positions and Targets' + real_data_name[11:])
plt.show()

In [ ]:
# sim 与 real 对比
data_root = '/home/fourier/GRX_humanoid/'
sdk_path = data_root + 'real data/251115_数采log_replay/'
sim_data_name = 'wbc_full38_0.csv'
sim_data_path = sdk_path + '/mujoco测试/' + sim_data_name
real_data_name = 'wbc_full14_251125_163042.csv'
real_data_path = sdk_path + '/aurora/' + real_data_name

sim_csv = pd.read_csv(sim_data_path)
real_csv = pd.read_csv(real_data_path)

print(sim_csv.shape)
print(real_csv.shape)

In [ ]:
sim_start_id = 282
time_strings = sim_csv.iloc[sim_start_id:, 0]
sim_time = [datetime.strptime(t[11:], '%H:%M:%S.%f') for t in time_strings]
sim_time_seconds = [(t - sim_time[0]).total_seconds() for t in sim_time]
print(sim_time_seconds[:10])
print(len(sim_time_seconds))

real_start_id = 0
real_time = real_csv.iloc[real_start_id:, 0]
real_time_seconds = [(t - real_time[0]) for t in real_time]
real_time_seconds = np.round(real_time_seconds, decimals=4)
print(len(real_time_seconds))
print(real_time_seconds[:10])

In [ ]:
# parse data
ac_dim = 31
upper_num = 16
sim_data = sim_csv.iloc[sim_start_id:, :].values
sim_ang_vel = sim_data[:, 1:4]
sim_gravity = sim_data[:, 4:7]
sim_jpos = sim_data[:, 7:7+ac_dim]
sim_jvel = sim_data[:, 7+ac_dim:7+2*ac_dim]
sim_jTor = sim_data[:, 7+2*ac_dim:7+3*ac_dim]
sim_jTarg = sim_data[:, 7+3*ac_dim:7+4*ac_dim]
sim_action = sim_data[:, 7+4*ac_dim:7+5*ac_dim]
sim_walk_flag = sim_data[:, 7+5*ac_dim:7+5*ac_dim+1]
sim_velCmd = sim_data[:, 7+5*ac_dim+1:7+5*ac_dim+4]
sim_haCmd = sim_data[:, 7+5*ac_dim+4:7+5*ac_dim+4+4]
sim_upper_cmd = sim_data[:, 7+5*ac_dim+4+4:7+5*ac_dim+4+4+upper_num]
sim_CoM_pos = sim_data[:, 7+5*ac_dim+4+4+upper_num:7+5*ac_dim+4+4+upper_num+3]

real_data = real_csv.iloc[real_start_id:, :].values
real_ang_vel = real_data[:, 1:4]
real_gravity = real_data[:, 4:7]
real_jpos = real_data[:, 7:7+ac_dim]
real_jvel = real_data[:, 7+ac_dim:7+2*ac_dim]
real_jTor = real_data[:, 7+2*ac_dim:7+3*ac_dim]
real_jTarg = real_data[:, 7+3*ac_dim:7+4*ac_dim]
real_action = real_data[:, 7+4*ac_dim:7+5*ac_dim]
real_walk_flag = real_data[:, 7+5*ac_dim:7+5*ac_dim+1]
real_velCmd = real_data[:, 7+5*ac_dim+1:7+5*ac_dim+4]
real_haCmd = real_data[:, 7+5*ac_dim+4:7+5*ac_dim+4+4]
real_upper_cmd = real_data[:, 7+5*ac_dim+4+4:7+5*ac_dim+4+4+upper_num]
real_CoM_pos = real_data[:, 7+5*ac_dim+4+4+upper_num:7+5*ac_dim+4+4+upper_num+3]

In [ ]:
# joint pos comparison
#  leg joints
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(sim_time_seconds, sim_jpos[:, i], label=f'sim_jPos')
    ax.plot(real_time_seconds, real_jpos[:, i], label=f'real_jPos')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(sim_time_seconds, sim_haCmd[:, 0], label=f'hgt_cmd', linestyle='--')
    ax2.plot(sim_time_seconds, sim_haCmd[:, 1], label=f'yaw_cmd', linestyle='--')
    ax2.plot(sim_time_seconds, sim_haCmd[:, 2], label=f'roll_cmd', linestyle='--')
    ax2.plot(sim_time_seconds, sim_haCmd[:, 3], label=f'pitch_cmd', linestyle='-.')
    # ax2.set_ylabel('HA Command (m/s or rad/s)')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Lower Body Joint Position Comparison', fontsize=16)
plt.show()

# upper joints
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16):
    ax = axs[i // 4, i % 4]
    ax.plot(sim_time_seconds, sim_jpos[:, i+15], label=f'sim_jPos')
    ax.plot(sim_time_seconds, sim_upper_cmd[:, i], label=f'sim_upper_cmd', linestyle='--')
    ax.plot(real_time_seconds, real_jpos[:, i+15], label=f'real_jPos')
    ax.plot(real_time_seconds, real_upper_cmd[:, i], label=f'real_upper_cmd', linestyle='--')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Upper Body Joint Position Comparison', fontsize=16)
plt.show()

In [ ]:
# joint vel comparison
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(sim_time_seconds, sim_jvel[:, i], label=f'sim_jVel')
    ax.plot(real_time_seconds, real_jvel[:, i], label=f'real_jVel')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(sim_time_seconds, sim_ang_vel[:, 0], label=f'sim_Wx', linestyle='--')
    ax2.plot(sim_time_seconds, sim_ang_vel[:, 1], label=f'sim_Wy', linestyle='--')
    ax2.plot(sim_time_seconds, sim_ang_vel[:, 2], label=f'sim_Wz', linestyle='--')
    ax2.plot(real_time_seconds, real_ang_vel[:, 0], label=f'real_Wx', linestyle='-.')
    ax2.plot(real_time_seconds, real_ang_vel[:, 1], label=f'real_Wy', linestyle='-.')
    ax2.plot(real_time_seconds, real_ang_vel[:, 2], label=f'real_Wz', linestyle='-.')
    ax2.set_ylabel('Angular Velocity (rad/s)')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Lower Body Joint Velocity Comparison', fontsize=16)
plt.show()

fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16):
    ax = axs[i // 4, i % 4]
    ax.plot(sim_time_seconds, sim_jvel[:, i+15], label=f'sim_jVel')
    ax.plot(real_time_seconds, real_jvel[:, i+15], label=f'real_jVel')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(sim_time_seconds, sim_ang_vel[:, 0], label=f'sim_Wx', linestyle='--')
    ax2.plot(sim_time_seconds, sim_ang_vel[:, 1], label=f'sim_Wy', linestyle='--')
    ax2.plot(sim_time_seconds, sim_ang_vel[:, 2], label=f'sim_Wz', linestyle='--')
    ax2.plot(real_time_seconds, real_ang_vel[:, 0], label=f'real_Wx', linestyle='-.')
    ax2.plot(real_time_seconds, real_ang_vel[:, 1], label=f'real_Wy', linestyle='-.')
    ax2.plot(real_time_seconds, real_ang_vel[:, 2], label=f'real_Wz', linestyle='-.')
    ax2.set_ylabel('Angular Velocity (rad/s)')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Upper Body Joint Velocity Comparison', fontsize=16)
plt.show()

In [ ]:
# joint torque comparison
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(sim_time_seconds, sim_jTor[:, i], label=f'sim_jTor')
    ax.plot(real_time_seconds, real_jTor[:, i], label=f'real_jTor')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Lower Body Joint Torque Comparison', fontsize=16)
plt.show()

fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16): 
    ax = axs[i // 4, i % 4]
    ax.plot(sim_time_seconds, sim_jTor[:, i+15], label=f'sim_jTor')
    ax.plot(real_time_seconds, real_jTor[:, i+15], label=f'real_jTor')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Upper Body Joint Torque Comparison', fontsize=16)
plt.show()

In [ ]:
# 三段
data_root = '/home/fourier/GRX_humanoid/'
sdk_path = data_root + 'real data/251115_数采log_replay'
real_data_name = '28号摩擦力补偿测试wbc_full38_0.csv'
real_data_path = sdk_path + '/' + real_data_name

start_id = 12665-1
M = 22387-12665-1
real_data_no_comp = pd.read_csv(real_data_path, 
                                skiprows=range(1, start_id + 1),
                                nrows=M)

start_id = 22387-1
M = 32455-22387-1

# real_data_no_comp = pd.read_csv(real_data_path, 
#                                 skiprows=range(1, start_id + 1),
#                                 nrows=M).values

start_id = 32455
real_data_arm_comp = pd.read_csv(real_data_path, 
                                skiprows=range(1, start_id))

print(real_data_no_comp.shape)
print(real_data_arm_comp.shape)

In [ ]:
no_comp_start_id = 0
time_strings = real_data_no_comp.iloc[no_comp_start_id:, 0]
no_comp_time = [datetime.strptime(t[11:], '%H:%M:%S.%f') for t in time_strings]
no_comp_time_seconds = [(t - no_comp_time[0]).total_seconds() for t in no_comp_time]
print(no_comp_time_seconds[:10])

arm_comp_start_id = 0
arm_comp_time_strings = real_data_arm_comp.iloc[arm_comp_start_id:, 0]
arm_comp_time = [datetime.strptime(t[11:], '%H:%M:%S.%f') for t in arm_comp_time_strings]
arm_comp_time_seconds = [(t - arm_comp_time[0]).total_seconds() for t in arm_comp_time]
print(arm_comp_time_seconds[:10])

In [ ]:
# parse data
ac_dim = 31
upper_num = 16
no_comp_data = real_data_no_comp.iloc[:, :].values
no_comp_ang_vel = no_comp_data[:, 1:4]
no_comp_gravity = no_comp_data[:, 4:7]
no_comp_jpos = no_comp_data[:, 7:7+ac_dim]
no_comp_jvel = no_comp_data[:, 7+ac_dim:7+2*ac_dim]
no_comp_jTor = no_comp_data[:, 7+2*ac_dim:7+3*ac_dim]
no_comp_jTarg = no_comp_data[:, 7+3*ac_dim:7+4*ac_dim]
no_comp_action = no_comp_data[:, 7+4*ac_dim:7+5*ac_dim]
no_comp_walk_flag = no_comp_data[:, 7+5*ac_dim:7+5*ac_dim+1]
no_comp_velCmd = no_comp_data[:, 7+5*ac_dim+1:7+5*ac_dim+4]
no_comp_haCmd = no_comp_data[:, 7+5*ac_dim+4:7+5*ac_dim+4+4]
no_comp_upper_cmd = no_comp_data[:, 7+5*ac_dim+4+4:7+5*ac_dim+4+4+upper_num]
no_comp_CoM_pos = no_comp_data[:, 7+5*ac_dim+4+4+upper_num:7+5*ac_dim+4+4+upper_num+3]

arm_comp_data = real_data_arm_comp.iloc[:, :].values
arm_comp_ang_vel = arm_comp_data[:, 1:4]
arm_comp_gravity = arm_comp_data[:, 4:7]
arm_comp_jpos = arm_comp_data[:, 7:7+ac_dim]
arm_comp_jvel = arm_comp_data[:, 7+ac_dim:7+2*ac_dim]
arm_comp_jTor = arm_comp_data[:, 7+2*ac_dim:7+3*ac_dim]
arm_comp_jTarg = arm_comp_data[:, 7+3*ac_dim:7+4*ac_dim]
arm_comp_action = arm_comp_data[:, 7+4*ac_dim:7+5*ac_dim]
arm_comp_walk_flag = arm_comp_data[:, 7+5*ac_dim:7+5*ac_dim+1]
arm_comp_velCmd = arm_comp_data[:, 7+5*ac_dim+1:7+5*ac_dim+4]
arm_comp_haCmd = arm_comp_data[:, 7+5*ac_dim+4:7+5*ac_dim+4+4]
arm_comp_upper_cmd = arm_comp_data[:, 7+5*ac_dim+4+4:7+5*ac_dim+4+4+upper_num]
arm_comp_CoM_pos = arm_comp_data[:, 7+5*ac_dim+4+4+upper_num:7+5*ac_dim+4+4+upper_num+3]

In [ ]:
# joint pos comparison
#  leg joints
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(no_comp_time_seconds, no_comp_jpos[:, i], label=f'no_comp_jPos')
    ax.plot(arm_comp_time_seconds, arm_comp_jpos[:, i], label=f'arm_comp_jPos')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Lower Body Joint Position Comparison', fontsize=16)
plt.show()

# upper joints
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16):
    ax = axs[i // 4, i % 4]
    ax.plot(no_comp_time_seconds, no_comp_jpos[:, i+15], label=f'no_comp_jPos')
    ax.plot(no_comp_time_seconds, no_comp_upper_cmd[:, i], label=f'no_comp_upper_cmd', linestyle='--')
    ax.plot(arm_comp_time_seconds, arm_comp_jpos[:, i+15], label=f'arm_comp_jPos', linestyle='--')
    ax.plot(arm_comp_time_seconds, arm_comp_upper_cmd[:, i], label=f'arm_comp_upper_cmd', linestyle='--')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Upper Body Joint Position Comparison', fontsize=16)
plt.show()

In [ ]:
# joint vel comparison
ang_vel_label = ['Wx', 'Wy', 'Wz']
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(no_comp_time_seconds, no_comp_jvel[:, i], label=f'no_comp_jVel')
    ax.plot(arm_comp_time_seconds, arm_comp_jvel[:, i], label=f'arm_comp_jVel')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 0], label=f'no_comp_Wx', linestyle='--')
    ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 1], label=f'no_comp_Wy', linestyle='--')
    ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 2], label=f'no_comp_Wz', linestyle='--')
    ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 0], label=f'arm_comp_Wx', linestyle='-.')
    ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 1], label=f'arm_comp_Wy', linestyle='-.')
    ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 2], label=f'arm_comp_Wz', linestyle='-.')
    ax2.set_ylabel('Angular Velocity (rad/s)')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Lower Body Joint Velocity Comparison', fontsize=16)
plt.show()

fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16):
    ax = axs[i // 4, i % 4]
    if i < 13:
        ax.plot(no_comp_time_seconds, no_comp_jvel[:, i+15+2], label=f'no_comp_jVel')
        ax.plot(arm_comp_time_seconds, arm_comp_jvel[:, i+15+2], label=f'arm_comp_jVel')
        ax.set_title(upper_joint_names[i+2])
        ax.legend(loc = 'upper left')
    else:
        ax.plot(no_comp_time_seconds, no_comp_ang_vel[:, 0 + i-13], label=f'no_comp_{ang_vel_label[i-13]}', linestyle='--')
        ax.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 0 + i-13], label=f'arm_comp_{ang_vel_label[i-13]}', linestyle='-.')
        ax.set_title(ang_vel_label[i-13])
        ax.legend(loc = 'upper left')
    # ax2 = ax.twinx()
    # ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 0], label=f'no_comp_Wx', linestyle='--')
    # ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 1], label=f'no_comp_Wy', linestyle='--')
    # ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 2], label=f'no_comp_Wz', linestyle='--')
    # ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 0], label=f'arm_comp_Wx', linestyle='-.')
    # ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 1], label=f'arm_comp_Wy', linestyle='-.')
    # ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 2], label=f'arm_comp_Wz', linestyle='-.')
    # ax2.set_ylabel('Angular Velocity (rad/s)')
    # ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Upper Body Joint Velocity Comparison', fontsize=16)
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16):
    ax = axs[i // 4, i % 4]
    if i < 13:
        ax.plot(no_comp_time_seconds, no_comp_jvel[:, i+15+2], label=f'no_comp_jVel')
        ax.plot(arm_comp_time_seconds, arm_comp_jvel[:, i+15+2], label=f'arm_comp_jVel')
        ax.set_title(upper_joint_names[i+2])
        ax.legend(loc = 'upper left')
    else:
        ax.plot(no_comp_time_seconds, no_comp_ang_vel[:, 0 + i-13], label=f'no_comp_{ang_vel_label[i-13]}', linestyle='--')
        ax.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 0 + i-13], label=f'arm_comp_{ang_vel_label[i-13]}', linestyle='-.')
        ax.set_title(ang_vel_label[i-13])
        ax.legend(loc = 'upper left')
    # ax2 = ax.twinx()
    # ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 0], label=f'no_comp_Wx', linestyle='--')
    # ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 1], label=f'no_comp_Wy', linestyle='--')
    # ax2.plot(no_comp_time_seconds, no_comp_ang_vel[:, 2], label=f'no_comp_Wz', linestyle='--')
    # ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 0], label=f'arm_comp_Wx', linestyle='-.')
    # ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 1], label=f'arm_comp_Wy', linestyle='-.')
    # ax2.plot(arm_comp_time_seconds, arm_comp_ang_vel[:, 2], label=f'arm_comp_Wz', linestyle='-.')
    # ax2.set_ylabel('Angular Velocity (rad/s)')
    # ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.suptitle('Upper Body Joint Velocity Comparison', fontsize=16)
plt.show()

In [ ]:
# joint torque comparison
fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(15):
    ax = axs[i // 4, i % 4]
    ax.plot(no_comp_time_seconds, no_comp_jTor[:, i], label=f'no_comp_jTor')
    ax.plot(arm_comp_time_seconds, arm_comp_jTor[:, i], label=f'arm_comp_jTor')
    ax.set_title(lower_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Lower Body Joint Torque Comparison', fontsize=16)
plt.show()

fig, axs = plt.subplots(4, 4, figsize=(40, 20), sharex=True)
for i in range(16): 
    ax = axs[i // 4, i % 4]
    ax.plot(no_comp_time_seconds, no_comp_jTor[:, i+15], label=f'no_comp_jTor')
    ax.plot(arm_comp_time_seconds, arm_comp_jTor[:, i+15], label=f'arm_comp_jTor')
    ax.set_title(upper_joint_names[i])
    ax.legend(loc = 'upper left')
plt.tight_layout()
plt.suptitle('Upper Body Joint Torque Comparison', fontsize=16)
plt.show()